# Derm-Vision: ISIC 2019 Skin Lesion Classification — Training

This notebook runs the full training pipeline on Google Colab with GPU.

Data is downloaded directly from Kaggle to Colab's local disk — no Google Drive storage needed for the dataset.

## 1. GPU Check

In [ ]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    !nvidia-smi --query-gpu=memory.total --format=csv,noheader

GPU available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
97887 MiB


## 2. Clone Repo & Install Dependencies

In [ ]:
!git clone https://github.com/Lambert-Nguyen/derm-vision.git /content/derm-vision
%cd /content/derm-vision

Cloning into '/content/derm-vision'...
remote: Enumerating objects: 104, done.
remote: Counting objects: 100% (104/104), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 104 (delta 38), reused 86 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (104/104), 5.31 MiB | 31.09 MiB/s, done.
Resolving deltas: 100% (38/38), done.
/content/derm-vision


In [ ]:
!pip install -q timm albumentations wandb grad-cam kaggle ttach

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 69.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 3. Download ISIC 2019 from Kaggle

Downloads directly to Colab's local disk (~100GB free) — no Drive space needed.

**Setup**: Go to https://www.kaggle.com/settings → copy your **Username** and **API Key**, then paste them below.

In [ ]:
import os
import json
from getpass import getpass

# Prompt for credentials (won't be displayed or saved in the notebook)
kaggle_username = input("Kaggle username: ")
kaggle_key = getpass("Kaggle API key: ")

# Write kaggle.json
kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)
kaggle_json_path = os.path.join(kaggle_dir, "kaggle.json")
with open(kaggle_json_path, "w") as f:
    json.dump({"username": kaggle_username, "key": kaggle_key}, f)
os.chmod(kaggle_json_path, 0o600)
print("Kaggle API configured.")

Kaggle username: duylam1407
Kaggle API key: ··········
Kaggle API configured.


In [ ]:
# Download ISIC 2019 dataset
!kaggle datasets download -d andrewmvd/isic-2019 -p /content/isic-data --unzip

# List downloaded files
!ls /content/isic-data/

Dataset URL: https://www.kaggle.com/datasets/andrewmvd/isic-2019
License(s): Attribution-NonCommercial 4.0 International (CC BY-NC 4.0)
100% 9.10G/9.10G [00:47<00:00, 206MB/s]

ISIC_2019_Training_GroundTruth.csv  ISIC_2019_Training_Metadata.csv
ISIC_2019_Training_Input


In [ ]:
# Symlink data into the project directory
import os
import glob

os.makedirs("data/raw", exist_ok=True)

# Auto-detect the data layout from Kaggle
kaggle_dir = "/content/isic-data"

# Find the images directory
img_dir = None
for candidate in [
    os.path.join(kaggle_dir, "ISIC_2019_Training_Input"),
    os.path.join(kaggle_dir, "ISIC_2019_Training_Input", "ISIC_2019_Training_Input"),
]:
    if os.path.isdir(candidate) and glob.glob(os.path.join(candidate, "*.jpg")):
        img_dir = candidate
        break

if img_dir is None:
    raise FileNotFoundError(
        f"Could not find images in {kaggle_dir}. Contents: {os.listdir(kaggle_dir)}"
    )

# Find CSV files
gt_matches = glob.glob(os.path.join(kaggle_dir, "**", "*GroundTruth*"), recursive=True)
meta_matches = glob.glob(os.path.join(kaggle_dir, "**", "*Metadata*"), recursive=True)
if not gt_matches or not meta_matches:
    raise FileNotFoundError(f"Missing CSVs in {kaggle_dir}. Contents: {os.listdir(kaggle_dir)}")
gt_csv = gt_matches[0]
meta_csv = meta_matches[0]

# Create symlinks
links = [
    (img_dir, "data/raw/ISIC_2019_Training_Input/ISIC_2019_Training_Input"),
    (gt_csv, "data/raw/ISIC_2019_Training_GroundTruth.csv"),
    (meta_csv, "data/raw/ISIC_2019_Training_Metadata.csv"),
]

for src, dst in links:
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if os.path.exists(dst) or os.path.islink(dst):
        os.remove(dst)
    os.symlink(src, dst)
    print(f"{dst} -> {src}")

# Verify
n_images = len(glob.glob("data/raw/ISIC_2019_Training_Input/ISIC_2019_Training_Input/*.jpg"))
print(f"\nFound {n_images} images.")

data/raw/ISIC_2019_Training_Input/ISIC_2019_Training_Input -> /content/isic-data/ISIC_2019_Training_Input/ISIC_2019_Training_Input
data/raw/ISIC_2019_Training_GroundTruth.csv -> /content/isic-data/ISIC_2019_Training_GroundTruth.csv
data/raw/ISIC_2019_Training_Metadata.csv -> /content/isic-data/ISIC_2019_Training_Metadata.csv

Found 25331 images.


## 4. Create Data Splits

In [ ]:
if not os.path.exists("data/splits/train.csv"):
    !python scripts/create_splits.py
else:
    print("Splits already exist, skipping.")

Splits already exist, skipping.


## 5. Mount Drive for Checkpoints Only

Checkpoints (~50MB) are saved to Drive so they persist across sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import yaml

# Read config, update output paths only, rewrite preserving structure
config_path = "configs/config.yaml"
with open(config_path) as f:
    config_text = f.read()
    cfg = yaml.safe_load(config_text)

# Save checkpoints & results to Drive (small files only)
drive_output = "/content/drive/MyDrive/derm-vision-outputs"
os.makedirs(os.path.join(drive_output, "checkpoints"), exist_ok=True)
os.makedirs(os.path.join(drive_output, "results"), exist_ok=True)

# Replace only the output paths via string replacement to preserve comments
config_text = config_text.replace(
    f"checkpoint_dir: {cfg['output']['checkpoint_dir']}",
    f"checkpoint_dir: {os.path.join(drive_output, 'checkpoints')}"
)
config_text = config_text.replace(
    f"results_dir: {cfg['output']['results_dir']}",
    f"results_dir: {os.path.join(drive_output, 'results')}"
)

with open(config_path, "w") as f:
    f.write(config_text)

# Reload to verify
with open(config_path) as f:
    cfg = yaml.safe_load(f)
print("Checkpoints will save to Drive (minimal storage needed).")
print(f"  checkpoint_dir: {cfg['output']['checkpoint_dir']}")
print(f"  results_dir: {cfg['output']['results_dir']}")

Checkpoints will save to Drive (minimal storage needed).
  checkpoint_dir: /content/drive/MyDrive/derm-vision-outputs/checkpoints
  results_dir: /content/drive/MyDrive/derm-vision-outputs/results


## 6. Login to W&B

In [ ]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: nguyenphuongduylam (nguyenphuongduylam-san-jose-state-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 7. Train

In [ ]:
!python -m src.train --config configs/config.yaml

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: nguyenphuongduylam (nguyenphuongduylam-san-jose-state-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
]11;?]11;?wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /content/derm-vision/wandb/run-20260404_004430-5svlyefo
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run usual-salad-4
wandb: ⭐️ View project at https://wandb.ai/nguyenphuongduylam-san-jose-state-university/derm-vision
wandb: 🚀 View run at https://wandb.ai/nguyenphuongduylam-san-jose-state-university/derm-vision/runs/5svlyefo
model.safetensors: 100% 49.3M/49.3M [00:00<00:00, 60.4MB/s]
Using Focal Loss (gamma=2.0) with class weights
Epoch 1/50 | Train Loss: 1.5388 | Val Loss: 1.5297 | Val Bal-Acc: 0.3568
  Saved best 

## 8. Evaluate on Test Set (with Test-Time Augmentation)

In [ ]:
import ttach
import yaml
import torch
from src.dataset import ISICDataset, CLASS_NAMES
from src.transforms import get_val_transforms
from src.models.efficientnet import EfficientNetB3Classifier
from src.evaluate import compute_metrics, print_classification_report, plot_confusion_matrix
from torch.utils.data import DataLoader

with open("configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load best model
model = EfficientNetB3Classifier(
    num_classes=cfg["data"]["num_classes"],
    pretrained=False,
    dropout=cfg["model"]["dropout"],
).to(device)
ckpt_path = os.path.join(cfg["output"]["checkpoint_dir"], "best_model.pth")
model.load_state_dict(torch.load(ckpt_path, map_location=device))
model.eval()

# Wrap with test-time augmentation (flips + rotations, averaged)
tta_model = ttach.ClassificationTTAWrapper(
    model,
    ttach.aliases.d4_transform(),  # 8 variants: flips + 90° rotations
    merge_mode="mean",
)

# Test dataset
test_dataset = ISICDataset(
    image_dir=cfg["data"]["data_dir"],
    labels_csv=os.path.join(cfg["data"]["splits_dir"], "test.csv"),
    transform=get_val_transforms(cfg["data"]["image_size"]),
)
test_loader = DataLoader(test_dataset, batch_size=cfg["training"]["batch_size"], num_workers=2)

# --- Evaluate WITH TTA ---
print("=" * 50)
print("Results WITH Test-Time Augmentation (D4)")
print("=" * 50)
all_preds_tta, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        outputs = tta_model(images.to(device))
        all_preds_tta.extend(outputs.argmax(dim=1).cpu().tolist())
        all_labels.extend(labels.tolist())

metrics_tta = compute_metrics(all_labels, all_preds_tta)
print(f"Balanced Accuracy: {metrics_tta['balanced_accuracy']:.4f}")
print(f"Weighted F1:       {metrics_tta['weighted_f1']:.4f}")
print()
print_classification_report(all_labels, all_preds_tta)

save_path = os.path.join(cfg["output"]["results_dir"], "confusion_matrix_tta.png")
plot_confusion_matrix(all_labels, all_preds_tta, save_path=save_path)

# --- Evaluate WITHOUT TTA (for comparison) ---
print("\n" + "=" * 50)
print("Results WITHOUT TTA (baseline)")
print("=" * 50)
all_preds_base = []
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images.to(device))
        all_preds_base.extend(outputs.argmax(dim=1).cpu().tolist())

metrics_base = compute_metrics(all_labels, all_preds_base)
print(f"Balanced Accuracy: {metrics_base['balanced_accuracy']:.4f}")
print(f"Weighted F1:       {metrics_base['weighted_f1']:.4f}")

# Summary
print("\n" + "=" * 50)
print("TTA Improvement")
print("=" * 50)
print(f"Balanced Accuracy: {metrics_base['balanced_accuracy']:.4f} → {metrics_tta['balanced_accuracy']:.4f} ({metrics_tta['balanced_accuracy'] - metrics_base['balanced_accuracy']:+.4f})")
print(f"Weighted F1:       {metrics_base['weighted_f1']:.4f} → {metrics_tta['weighted_f1']:.4f} ({metrics_tta['weighted_f1'] - metrics_base['weighted_f1']:+.4f})")

Results WITH Test-Time Augmentation (D4)
Balanced Accuracy: 0.8215
Weighted F1:       0.7102

              precision    recall  f1-score   support

         MEL       0.45      0.88      0.60       452
          NV       0.97      0.56      0.71      1288
         BCC       0.87      0.86      0.86       333
       AKIEC       0.66      0.72      0.69        86
         BKL       0.60      0.79      0.68       263
          DF       0.69      0.92      0.79        24
        VASC       0.81      1.00      0.89        25
         SCC       0.66      0.84      0.74        63

    accuracy                           0.70      2534
   macro avg       0.71      0.82      0.75      2534
weighted avg       0.80      0.70      0.71      2534

Confusion matrix saved to /content/drive/MyDrive/derm-vision-outputs/results/confusion_matrix_tta.png

Results WITHOUT TTA (baseline)
Balanced Accuracy: 0.8195
Weighted F1:       0.7013

TTA Improvement
Balanced Accuracy: 0.8195 → 0.8215 (+0.0020)
Weighte